# Exercise 30.1: Implementing state-based cooperativity

In this exercise, we will implement the Razumova (2000) stiffness-distortion crossbridge model.

This model tracks the proportion of regulatory units (RUs) in four distinct functional states:

1. $R_{\mathrm{off}}$: The thin filament is inactive.
2. $D$: Detached state (thin filament active, but myosin not bound).
3. $A_1$: Attached crossbridge, pre-powerstroke (no force).
4. $A_2$: Attached crossbridge, post-powerstroke (force-generating).

The true power of this model is that the transition rates between these states dynamically change based on **cooperativity**.


## Exercise 30.1a: Defining the cooperative ODEs

We will define the right-hand side (RHS) of the system for `solve_ivp`.

Your task is to translate the mathematical cooperativity equations into Python code, and then implement the mass-action ordinary differential equations.

Recall the cooperativity multipliers:

- **RU-RU ($u$):** $k_{\mathrm{on}}^w = k_{\mathrm{on}}^u \left[1 + \lambda^{\mathrm{on}}(u - 1)\right]^2$
- **XB-RU ($w$):** $k_{\mathrm{on}} = k_{\mathrm{on}}^w \left[1 + \lambda^{A_2}(e^{w-1} - 1)\right]^2$
- **XB-XB ($v$):** $f = f_0 \left[1 + \lambda^{A_2}(e^{v-1} - 1)\right]^2$

_(Note: The reverse rates $k_{\mathrm{off}}$ and $f'$ have similar forms, but with negative exponents or flipped signs. They have been provided for you below to save time).\_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


def rhs_razumova(t, y, k_u_on, k_u_off, f_0, f_prime_0, h, h_prime, g, u, v, w):
    # Unpack the states
    D, A1, A2 = y

    # 1. Mass conservation (Assume total RUs = 1.0)
    R_off = 1.0 - (D + A1 + A2)

    # 2. Define the active subpopulations
    lam_on = D + A1 + A2
    lam_A2 = A2

    # 3. Implement RU-RU Cooperativity (Parameter u)
    k_w_on = ...
    k_w_off = k_u_off * (u - lam_on * (u - 1)) ** 2

    # 4. Implement XB-RU Cooperativity (Parameter w)
    k_on = ...
    k_off = k_w_off * (1 + lam_A2 * (np.exp(-(w - 1)) - 1)) ** 2

    # 5. Implement XB-XB Cooperativity (Parameter v)
    f = ...
    f_prime = f_prime_0 * (1 + lam_A2 * (np.exp(-(v - 1)) - 1)) ** 2

    # 6. Calculate the derivatives using mass-action
    # Use the diagram from the text to write the ODEs!
    dD_dt = ...
    dA1_dt = ...
    dA2_dt = ...

    return [dD_dt, dA1_dt, dA2_dt]

## Exercise 30.1b: Simulating force development

We will now solve the system. Because $A_2$ represents the post-powerstroke crossbridges, the proportion of units in the $A_2$ state serves as a direct proxy for the macroscopic force produced by the muscle.

Try changing the cooperativity parameters ($u$, $v$, and $w$) in the code below!

- Set them all to 1.0 to see the "baseline" system with no cooperativity.
- Increase them to 2.0 or 3.0 to see how the proteins "help" each other activate faster and produce more total force.


In [ ]:
# Base kinetic parameters
k_u_on = 120.0
k_u_off = 50.0
f_0 = 50.0
f_prime_0 = 400.0
h_rate = 8.0
h_prime = 6.0
g_rate = 4.0

# Cooperativity parameters (Change these!)
u = 1.0  # RU-RU cooperativity
v = 1.0  # XB-XB cooperativity
w = 1.0  # XB-RU cooperativity

# Pack the parameters
params = (k_u_on, k_u_off, f_0, f_prime_0, h_rate, h_prime, g_rate, u, v, w)

# Initial conditions: almost everything is in R_off (D, A1, A2 ~ 0)
y0 = [0.01, 0.01, 0.01]
t_span = (0, 0.5)

# Solve the ODE
sol = solve_ivp(...)

# Plot the state probabilities
plt.plot(sol.t, sol.y[0], label="Detached ($D$)")
plt.plot(sol.t, sol.y[1], label="Pre-powerstroke ($A_1$)")
plt.plot(sol.t, sol.y[2], label="Post-powerstroke / Force ($A_2$)")
plt.show()